# WP4v5 — Notebook 4: Visualizations

Three figures:
1. **Native LLaVA vs projected MAE** — N images × 2-column grid (image | descriptions)
2. **MAE with and without masking** — same format, comparing descriptions with/without masking
3. **Visible patches + associated LLM embeddings** — masked image + visible patches colored by similarity

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from PIL import Image
from datasets import load_dataset
from transformers import (
    ViTImageProcessor, ViTMAEModel,
    LlavaForConditionalGeneration, CLIPImageProcessor, LlamaTokenizer
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PATCH_SIZE = 16
GRID = 14  # 224 / 16
print(f'Device : {DEVICE}')
import matplotlib.cm as cm

In [ ]:
import os
MAE_PATH = './vit-mae-large' if os.path.exists('./vit-mae-large') else 'facebook/vit-mae-large'
LLAVA_PATH = './llava-1.5-7b-hf' if os.path.exists('./llava-1.5-7b-hf') else 'llava-hf/llava-1.5-7b-hf'
# Auto-download ImageNet-100 if not found locally
DATASET_DIR = './imagenet100-hf' if os.path.exists('./imagenet100-hf/data') else './imagenet100'
if not os.path.exists(f'{DATASET_DIR}/data'):
    from huggingface_hub import snapshot_download
    DATASET_DIR = './imagenet100-hf'
    print('Downloading ImageNet-100 (~17 GB)...')
    snapshot_download(repo_id='ilee0022/ImageNet100', repo_type='dataset', local_dir=DATASET_DIR)
    print('Done.')

# ── Modèles ────────────────────────────────────────────────────────────────
class ProjectionMLP(nn.Module):
    def __init__(self, dim=1024):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, dim))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

f_theta = ProjectionMLP().to(DEVICE)
f_theta.load_state_dict(torch.load('wp4v5_ftheta_best.pt', map_location=DEVICE))
f_theta.eval()

norm   = torch.load('wp4v5_norm_stats.pt')
z_mean = norm['mean'].cpu()
z_std  = norm['std'].cpu()

mae_processor = ViTImageProcessor(
    size={'height': 224, 'width': 224},
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)
mae_encoder = ViTMAEModel.from_pretrained(MAE_PATH).to(DEVICE).eval()

llava_full   = LlavaForConditionalGeneration.from_pretrained(
    LLAVA_PATH, torch_dtype=torch.float16).to(DEVICE).eval()
llm          = llava_full.language_model
vision_tower = llava_full.vision_tower
mlp_conn     = llava_full.multi_modal_projector
clip_proc    = CLIPImageProcessor.from_pretrained(LLAVA_PATH)
tokenizer    = LlamaTokenizer.from_pretrained(LLAVA_PATH, use_fast=False)

SYSTEM    = ('A chat between a curious user and an artificial intelligence assistant. '
             'The assistant gives helpful, detailed, and polite answers to the user questions.')
USER_TEXT = 'Describe this image in one sentence.'
before_ids    = tokenizer(f'{SYSTEM} USER: ', return_tensors='pt', add_special_tokens=True).input_ids.to(DEVICE)
after_ids     = tokenizer(f'\n{USER_TEXT} ASSISTANT:', return_tensors='pt', add_special_tokens=False).input_ids.to(DEVICE)
before_embeds = llm.get_input_embeddings()(before_ids).half()
after_embeds  = llm.get_input_embeddings()(after_ids).half()

ds = load_dataset('parquet', data_files={'validation': f'{DATASET_DIR}/data/validation-*.parquet'})
print('Tout chargé')

In [ ]:
# ── Fonctions utilitaires ───────────────────────────────────────────────────
def project_tokens(tokens):
    z_norm = (tokens.cpu().float() - z_mean) / z_std
    with torch.no_grad():
        return f_theta(z_norm.to(DEVICE)).cpu().float()

def llm_describe(visual_tokens_clip, max_new_tokens=60):
    with torch.no_grad():
        visual_llm    = mlp_conn(visual_tokens_clip.half().to(DEVICE))
        inputs_embeds = torch.cat([before_embeds, visual_llm.unsqueeze(0), after_embeds], dim=1)
        out_ids       = llm.generate(
            inputs_embeds=inputs_embeds, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()

def describe_llava_native(image_pil, max_new_tokens=60):
    inp = clip_proc(images=image_pil, return_tensors='pt', do_rescale=True)
    pix = inp['pixel_values'].to(DEVICE).half()
    with torch.no_grad():
        vis           = vision_tower(pix).last_hidden_state[:, 1:]
        vis_llm       = mlp_conn(vis)[0]
        inputs_embeds = torch.cat([before_embeds, vis_llm.unsqueeze(0), after_embeds], dim=1)
        out_ids       = llm.generate(
            inputs_embeds=inputs_embeds, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()

def get_mae_tokens(image_pil, seed=None):
    """Retourne CLS (1024,), patch tokens (N, 1024), visible_patch_ids."""
    inp   = mae_processor(images=image_pil, return_tensors='pt')
    inp   = {k: v.to(DEVICE) for k, v in inp.items()}
    noise = torch.zeros(1, 196).to(DEVICE) if seed is None else \
            torch.rand(1, 196, generator=torch.Generator().manual_seed(seed)).to(DEVICE)
    with torch.no_grad():
        out = mae_encoder(**inp, noise=noise)
    hs          = out.last_hidden_state[0]
    visible_ids = torch.where(out.mask[0].cpu() == 0)[0].tolist()
    return hs[0].cpu().float(), hs[1:].cpu().float(), visible_ids

def wrap_text(text, max_chars=55):
    words, lines, line = text.split(), [], ''
    for w in words:
        if len(line) + len(w) + 1 > max_chars:
            lines.append(line); line = w
        else:
            line = (line + ' ' + w).strip()
    if line: lines.append(line)
    return '\n'.join(lines)

print('Fonctions définies')

## Figure 1 — Native LLaVA vs projected MAE

In [ ]:
N_IMAGES = 8
samples  = [ds['validation'][i] for i in range(N_IMAGES)]

print(f'{'Class':<25} {'LLaVA 1.5':<55} {'MAE projected'}')
print('-' * 130)
for s in samples:
    img         = s['image'].convert('RGB')
    label       = s['text'].split(',')[0].strip()
    desc_native = describe_llava_native(img)
    _, patches, _ = get_mae_tokens(img.resize((224, 224)), seed=None)
    desc_mae    = llm_describe(project_tokens(patches))
    print(f'{label:<25} {desc_native:<55} {desc_mae}')


## Figure 2 — MAE without masking vs with 75% masking

In [ ]:
MASK_SEED = 42
N_IMAGES  = 8
samples   = [ds['validation'][i] for i in range(N_IMAGES)]

print(f'{'Class':<25} {'MAE 196 patches':<55} {'MAE 49 patches (75% masked)'}')
print('-' * 130)
for s in samples:
    img   = s['image'].convert('RGB').resize((224, 224))
    label = s['text'].split(',')[0].strip()

    _, patches_full,   _           = get_mae_tokens(img, seed=None)
    _, patches_masked, visible_ids = get_mae_tokens(img, seed=MASK_SEED)

    desc_full   = llm_describe(project_tokens(patches_full))
    desc_masked = llm_describe(project_tokens(patches_masked))

    print(f'{label:<25} {desc_full:<55} {desc_masked}')


## Figure 3 — Visible patches and associated LLM embeddings

For each visible patch, its MAE embedding is projected into the LLM space
and the nearest LLM tokens are found in the LLM embedding matrix.

In [ ]:
IMG_IDX   = 0
SEED_VIZ  = 7
TOP_K     = 3   # top-k tokens LLM par patch

item      = ds['validation'][IMG_IDX]
image_pil = item['image'].convert('RGB').resize((224, 224))
label_txt = item['text']

_, patch_tokens, visible_ids = get_mae_tokens(image_pil, seed=SEED_VIZ)
n_visible = len(visible_ids)
print(f'Classe : {label_txt} | Patches visibles : {n_visible}')

# Projection patches → espace CLIP → espace LLM via mm_projector
with torch.no_grad():
    proj_clip = project_tokens(patch_tokens)              # (n_visible, 1024)
    proj_llm  = mlp_conn(proj_clip.half().to(DEVICE))     # (n_visible, 4096)
    proj_llm  = F.normalize(proj_llm.float(), dim=-1)

# Matrice d'embeddings du LLM
llm_emb_matrix = llm.get_input_embeddings().weight.data.float()  # (vocab_size, 4096)
llm_emb_norm   = F.normalize(llm_emb_matrix, dim=-1)             # normalisé pour cosinus

# Top-K tokens les plus proches pour chaque patch
sims     = proj_llm @ llm_emb_norm.T.to(DEVICE)                  # (n_visible, vocab_size)
topk_ids = sims.topk(TOP_K, dim=-1).indices.cpu()                 # (n_visible, TOP_K)
topk_tokens = [
    [tokenizer.decode([tid]).strip() for tid in row]
    for row in topk_ids.tolist()
]

# Score de similarité max par patch (pour colorisation)
max_sim = sims.max(dim=-1).values.cpu().numpy()
sim_norm = (max_sim - max_sim.min()) / (max_sim.max() - max_sim.min() + 1e-8)

print('Top-K tokens calculés')

In [ ]:
img_arr  = np.array(image_pil)

# ── Image masquée ──────────────────────────────────────────────────────────
img_masked = img_arr.copy().astype(float)
for pid in range(196):
    if pid not in visible_ids:
        r, c = pid // GRID, pid % GRID
        img_masked[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] = 80

# ── Layout ─────────────────────────────────────────────────────────────────
# Rangées de patches : on arrange les n_visible patches en grille de largeur COLS_PATCH
COLS_PATCH  = 8
rows_patch  = (n_visible + COLS_PATCH - 1) // COLS_PATCH

fig3_h = 4 + rows_patch * 1.8
fig3   = plt.figure(figsize=(14, fig3_h))
gs3    = gridspec.GridSpec(
    2, 1,
    height_ratios=[3.5, rows_patch * 1.8],
    hspace=0.35
)

# ── Rangée 1 : image originale + image masquée ─────────────────────────────
gs3_top = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs3[0], wspace=0.08)

ax_orig = fig3.add_subplot(gs3_top[0])
ax_orig.imshow(img_arr)
ax_orig.set_title('Original image', fontsize=9, fontweight='bold', pad=4)
ax_orig.axis('off')

ax_mask = fig3.add_subplot(gs3_top[1])
ax_mask.imshow(img_masked.astype(np.uint8))
ax_mask.set_title(f'75% random masking — {n_visible} visible patches (seed={SEED_VIZ})',
                  fontsize=9, fontweight='bold', pad=4)
# Encadrer les patches visibles en vert
for pid in visible_ids:
    r, c = pid // GRID, pid % GRID
    rect = plt.Rectangle(
        (c * PATCH_SIZE - 0.5, r * PATCH_SIZE - 0.5),
        PATCH_SIZE, PATCH_SIZE,
        linewidth=0.6, edgecolor='#2ecc71', facecolor='none'
    )
    ax_mask.add_patch(rect)
ax_mask.axis('off')

# Titre global
fig3.suptitle(f'Class: {label_txt} — Top-{TOP_K} nearest LLM tokens per visible patch',
              fontsize=10, fontweight='bold', y=1.01)

# ── Rangée 2 : grille des patches visibles + tokens LLM ────────────────────
gs3_bot = gridspec.GridSpecFromSubplotSpec(
    rows_patch, COLS_PATCH,
    subplot_spec=gs3[1],
    hspace=0.6, wspace=0.15
)

cmap = cm.get_cmap('RdYlGn')

for idx, pid in enumerate(visible_ids):
    row_p, col_p = idx // COLS_PATCH, idx % COLS_PATCH
    ax_p = fig3.add_subplot(gs3_bot[row_p, col_p])

    # Extrait du patch dans l'image originale
    pr, pc = pid // GRID, pid % GRID
    patch_crop = img_arr[
        pr*PATCH_SIZE:(pr+1)*PATCH_SIZE,
        pc*PATCH_SIZE:(pc+1)*PATCH_SIZE
    ]

    # Fond coloré selon similarité max
    color = cmap(sim_norm[idx])
    ax_p.set_facecolor(color)

    # Image du patch
    ax_p.imshow(patch_crop, extent=[0, 1, 0.35, 1], aspect='auto',
                interpolation='nearest', zorder=2)

    # Tokens LLM
    token_str = '  '.join(topk_tokens[idx])
    ax_p.text(0.5, 0.17, token_str,
              ha='center', va='center', fontsize=5.5,
              transform=ax_p.transAxes,
              bbox=dict(fc='white', ec='none', alpha=0.7, pad=1))

    ax_p.set_xlim(0, 1); ax_p.set_ylim(0, 1)
    ax_p.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax_p.spines.values():
        spine.set_edgecolor(cmap(sim_norm[idx]))
        spine.set_linewidth(1.5)

# Cacher les axes vides
for idx in range(len(visible_ids), rows_patch * COLS_PATCH):
    row_p, col_p = idx // COLS_PATCH, idx % COLS_PATCH
    fig3.add_subplot(gs3_bot[row_p, col_p]).axis('off')

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1))
cbar = fig3.colorbar(sm, ax=fig3.axes, shrink=0.3, pad=0.01, aspect=15)
cbar.set_label('Cosine similarity (normalized)', fontsize=7)

fig3.savefig('fig_patch_embeddings.pdf', bbox_inches='tight', dpi=150)
fig3.savefig('fig_patch_embeddings.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure 3 sauvegardée')